# 실험 결과 분석

모델 × 데이터셋 × 전략 3축 비교  
- **데이터**: `summary.json`, `trajectory_logs.jsonl`, `step_logs.jsonl`
- **비교 축**: 모델(phi35mini / qwen7bcoder), 데이터셋(humaneval / mbpp), 전략(single / repair / code_then_plan / code_then_plan_repair / policy_loop 등)

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 0. 경로 설정

In [2]:
RESULTS_ROOT = Path("../results")

# 존재하는 결과 폴더 자동 탐색
def discover_runs(results_root: Path) -> list[dict]:
    """
    results/<model>/<dataset>/<strategy>/summary.json 구조를 재귀 탐색.
    summary.json 이 있는 폴더를 하나의 run으로 간주.
    """
    runs = []
    for summary_path in sorted(results_root.rglob("summary.json")):
        parts = summary_path.parent.relative_to(results_root).parts
        if len(parts) >= 3:
            model, dataset, strategy = parts[0], parts[1], parts[2]
        elif len(parts) == 2:
            model, dataset, strategy = parts[0], parts[1], "unknown"
        else:
            continue
        runs.append({
            "model": model,
            "dataset": dataset,
            "strategy": strategy,
            "summary_path": summary_path,
            "dir": summary_path.parent,
        })
    return runs

runs = discover_runs(RESULTS_ROOT)
print(f"발견된 run 수: {len(runs)}")
for r in runs:
    print(f"  {r['model']:20s} | {r['dataset']:15s} | {r['strategy']}")

발견된 run 수: 11
  phi35mini            | humaneval       | repair
  phi35mini            | humaneval       | single
  qwen25coder7b        | humaneval       | code_then_plan
  qwen25coder7b        | humaneval       | code_then_plan_repair
  qwen25coder7b        | humaneval       | proposed_v1
  qwen25coder7b        | humaneval       | proposed_v2
  qwen25coder7b        | humaneval       | repair
  qwen25coder7b        | humaneval       | single
  qwen25coder7b        | mbpp            | code_then_plan
  qwen25coder7b        | mbpp            | proposed_v2
  qwen25coder7b        | mbpp            | single


## 1. summary.json 로드 → 핵심 지표 비교표

In [3]:
def load_summary(run: dict) -> dict:
    with open(run["summary_path"], encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "model":    run["model"],
        "dataset":  run["dataset"],
        "strategy": run["strategy"],
    }

    # 공통 지표
    row["total_problems"]         = data.get("total_problems", None)
    row["num_success"]            = data.get("num_success", None)
    row["success_at_k"]           = data.get("success_at_k", None)
    row["execution_success_rate"] = data.get("execution_success_rate", None)
    row["conditional_success"]    = data.get("conditional_success", None)
    row["max_calls"]              = data.get("max_calls", None)

    # extra_summary (failure breakdown)
    extra = data.get("extra_summary", {})
    row["code_failed"]         = extra.get("code_failed", None)
    row["define_test_failed"]  = extra.get("define_test_failed", None)
    row["run_test_failed"]     = extra.get("run_test_failed", None)

    # planning stats (code_then_plan_repair 계열)
    ps = data.get("planning_stats", {})
    row["plan_used"]              = ps.get("used_plan", None)
    row["repair_used"]            = ps.get("used_repair", None)
    row["planning_recovery_rate"] = ps.get("planning_recovery_rate", None)
    row["repair_recovery_rate"]   = ps.get("repair_recovery_rate", None)

    # policy_stats (policy_loop 계열)
    pol = data.get("policy_stats", {})
    pl = pol.get("problem_level", {})
    cl = pol.get("call_level", {})
    if pl:
        row["plan_used"]              = pl.get("plan_used_problems", row.get("plan_used"))
        row["repair_used"]            = pl.get("repair_used_problems", row.get("repair_used"))
        row["planning_recovery_rate"] = pl.get("plan_problem_recovery_rate", row.get("planning_recovery_rate"))
        row["repair_recovery_rate"]   = pl.get("repair_problem_recovery_rate", row.get("repair_recovery_rate"))
    if cl:
        row["planning_cycle_count"]        = cl.get("planning_cycle_count", None)
        row["planning_cycle_success_rate"] = cl.get("planning_cycle_success_rate", None)
        row["repair_call_count"]           = cl.get("repair_call_count", None)
        row["repair_call_success_rate"]    = cl.get("repair_call_success_rate", None)

    return row


summary_rows = [load_summary(r) for r in runs]
df_summary = pd.DataFrame(summary_rows)
df_summary

,model,dataset,strategy,total_problems,num_success,success_at_k,execution_success_rate,conditional_success,max_calls,code_failed,define_test_failed,run_test_failed,plan_used,repair_used,planning_recovery_rate,repair_recovery_rate,planning_cycle_count,planning_cycle_success_rate,repair_call_count,repair_call_success_rate
0,phi35mini,humaneval,repair,164,94,0.5732,0.9451,0.6065,20,9,0,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,phi35mini,humaneval,single,164,6,0.0366,0.0671,0.5455,1,153,0,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,qwen25coder7b,humaneval,code_then_plan,164,152,0.9268,0.9939,0.9325,20,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,qwen25coder7b,humaneval,code_then_plan_repair,164,151,0.9207,0.9756,0.9437,20,0,0,0,49.0000,20.0000,0.7347,0.1000,NaN,NaN,NaN,NaN
4,qwen25coder7b,humaneval,proposed_v1,164,154,0.9390,0.9939,0.9448,20,1,0,9,33.0000,22.0000,0.6364,0.5455,76.0000,0.2763,77.0000,0.1558
5,qwen25coder7b,humaneval,proposed_v2,164,153,0.9329,0.9939,0.9387,20,1,0,10,96.0000,109.0000,0.8750,0.3670,191.0000,0.4398,197.0000,0.2030
6,qwen25coder7b,humaneval,repair,164,138,0.8415,0.8780,0.9583,20,20,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,qwen25coder7b,humaneval,single,164,116,0.7073,0.8841,0.8000,1,19,0,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,qwen25coder7b,mbpp,code_then_plan,257,213,0.8288,0.9961,0.8320,20,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,qwen25coder7b,mbpp,proposed_v2,257,223,0.8677,0.9844,0.8814,20,4,0,30,119.0000,96.0000,0.7143,0.0208,394.0000,0.2157,244.0000,0.0082


### 1-1. 핵심 지표만 추린 비교표

In [4]:
KEY_COLS = [
    "model", "dataset", "strategy",
    "total_problems", "num_success",
    "success_at_k", "execution_success_rate", "conditional_success",
]
df_summary[KEY_COLS].sort_values(["dataset", "model", "strategy"])

,model,dataset,strategy,total_problems,num_success,success_at_k,execution_success_rate,conditional_success
0,phi35mini,humaneval,repair,164,94,0.5732,0.9451,0.6065
1,phi35mini,humaneval,single,164,6,0.0366,0.0671,0.5455
2,qwen25coder7b,humaneval,code_then_plan,164,152,0.9268,0.9939,0.9325
3,qwen25coder7b,humaneval,code_then_plan_repair,164,151,0.9207,0.9756,0.9437
4,qwen25coder7b,humaneval,proposed_v1,164,154,0.9390,0.9939,0.9448
5,qwen25coder7b,humaneval,proposed_v2,164,153,0.9329,0.9939,0.9387
6,qwen25coder7b,humaneval,repair,164,138,0.8415,0.8780,0.9583
7,qwen25coder7b,humaneval,single,164,116,0.7073,0.8841,0.8000
8,qwen25coder7b,mbpp,code_then_plan,257,213,0.8288,0.9961,0.8320
9,qwen25coder7b,mbpp,proposed_v2,257,223,0.8677,0.9844,0.8814


### 1-2. 전략별 피벗 (dataset × model 고정, 전략 비교)

In [5]:
for dataset in df_summary["dataset"].unique():
    sub = df_summary[df_summary["dataset"] == dataset]
    pivot = sub.pivot_table(
        index="model",
        columns="strategy",
        values="success_at_k",
    )
    print(f"\n=== {dataset} | success_at_k ===")
    display(pivot)


=== humaneval | success_at_k ===


strategy,code_then_plan,code_then_plan_repair,proposed_v1,proposed_v2,repair,single
model,,,,,,
phi35mini,NaN,NaN,NaN,NaN,0.5732,0.0366
qwen25coder7b,0.9268,0.9207,0.9390,0.9329,0.8415,0.7073



=== mbpp | success_at_k ===


strategy,code_then_plan,proposed_v2,single
model,,,
qwen25coder7b,0.8288,0.8677,0.5253


### 1-3. 모델별 피벗 (strategy × dataset)

In [6]:
for model in df_summary["model"].unique():
    sub = df_summary[df_summary["model"] == model]
    pivot = sub.pivot_table(
        index="strategy",
        columns="dataset",
        values=["success_at_k", "execution_success_rate", "conditional_success"],
    )
    print(f"\n=== {model} ===")
    display(pivot)


=== phi35mini ===


,conditional_success,execution_success_rate,success_at_k
dataset,humaneval,humaneval,humaneval
strategy,,,
repair,0.6065,0.9451,0.5732
single,0.5455,0.0671,0.0366



=== qwen25coder7b ===


conditional_success        execution_success_rate  \
dataset                         humaneval   mbpp              humaneval   
strategy                                                                  
code_then_plan                     0.9325 0.8320                 0.9939   
code_then_plan_repair              0.9437    NaN                 0.9756   
proposed_v1                        0.9448    NaN                 0.9939   
proposed_v2                        0.9387 0.8814                 0.9939   
repair                             0.9583    NaN                 0.8780   
single                             0.8000 0.7849                 0.8841   

                             success_at_k         
dataset                 mbpp    humaneval   mbpp  
strategy                                          
code_then_plan        0.9961       0.9268 0.8288  
code_then_plan_repair    NaN       0.9207    NaN  
proposed_v1              NaN       0.9390    NaN  
proposed_v2           0.9844       0.9329 0.8677  
repair                   NaN       0.8415    NaN  
single                0.6693       0.7073 0.5253

## 2. trajectory_logs.jsonl 로드 → 문제별 분석

In [7]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_trajectory_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "trajectory_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_traj = load_trajectory_logs(runs)
print(f"trajectory rows: {len(df_traj)}")
df_traj.head(3)

trajectory rows: 2083


/tmp/ipykernel_3467007/2220713918.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


,run_id,dataset,problem_id,method,trajectory_id,num_steps,call_count,final_status,failure_family,final_tests_passed,final_tests_total,total_tokens,total_latency,num_exec_fail,num_test_fail,transition_path,entropy_series,initial_avg_entropy,budget_used,model,strategy,total_input_tokens,total_output_tokens,used_plan,recovered_by,plan_recovery_attempt,used_repair,planning_recovery_attempt,planning_cycle_count,planning_cycle_call_cost,repair_call_count,stopped_by_policy,stop_reason,num_plan_calls,num_repair_calls,input_token_stats,output_token_stats,initial_status,initial_failure_family,initial_failed,success_at_call,calls_to_recovery
0,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,20,20,TEST_FAIL:AssertionError,TEST_FAIL,NaN,None,12624,36.7052,1.0000,19.0000,"[EXEC_FAIL:IndentationError, TEST_FAIL:Asserti...","[0.22266789208276247, 0.020086308756623678, 0....",0.2227,"{'tokens': 12624, 'calls': 20, 'latency': 36.7...",phi35mini,repair,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/1,repair_loop,HumanEval/1_run0,20,20,TEST_FAIL:AssertionError,TEST_FAIL,NaN,None,15772,64.4662,1.0000,19.0000,"[EXEC_FAIL:IndentationError, TEST_FAIL:Asserti...","[0.2612395826960043, 0.021891773377882593, 0.2...",0.2612,"{'tokens': 15772, 'calls': 20, 'latency': 64.4...",phi35mini,repair,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/2,repair_loop,HumanEval/2_run0,2,2,PASS,PASS,NaN,None,648,1.5624,1.0000,0.0000,"[EXEC_FAIL:IndentationError, PASS]","[0.3004276009427751, 0.05436650812236029]",0.3004,"{'tokens': 648, 'calls': 2, 'latency': 1.56238...",phi35mini,repair,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2-1. call_count 분포 (전략별 평균 호출 수)

In [8]:
if not df_traj.empty and "call_count" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy"])["call_count"]
        .agg(["mean", "median", "min", "max", "count"])
        .round(2)
        .sort_index()
    )

mean  median  min  max  count
model         dataset   strategy                                             
phi35mini     humaneval repair                9.7000  2.0000    1   20    164
                        single                1.0000  1.0000    1    1    164
qwen25coder7b humaneval code_then_plan        3.0000  1.0000    1   19    164
                        code_then_plan_repair 2.9700  1.0000    1   19    164
                        proposed_v1           2.4000  1.0000    1   20    164
                        proposed_v2           4.5300  3.0000    1   19    164
                        repair                4.3800  1.0000    1   20    164
                        single                1.0000  1.0000    1    1    164
              mbpp      code_then_plan        4.8400  1.0000    1   19    257
                        proposed_v2           5.0200  1.0000    1   20    257
                        single                1.0000  1.0000    1    1    257

### 2-2. 문제별 최종 상태 분포

In [9]:
if not df_traj.empty and "final_status" in df_traj.columns:
    status_col = "final_status"
    # coarse failure family
    df_traj["failure_family_coarse"] = df_traj[status_col].apply(
        lambda s: "PASS" if s == "PASS" else str(s).split(":")[0]
    )
    display(
        df_traj.groupby(["model", "dataset", "strategy", "failure_family_coarse"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

,model,dataset,strategy,failure_family_coarse,count
1,phi35mini,humaneval,repair,PASS,94
2,phi35mini,humaneval,repair,TEST_FAIL,61
0,phi35mini,humaneval,repair,EXEC_FAIL,9
3,phi35mini,humaneval,single,EXEC_FAIL,153
4,phi35mini,humaneval,single,PASS,6
5,phi35mini,humaneval,single,TEST_FAIL,5
7,qwen25coder7b,humaneval,code_then_plan,PASS,152
8,qwen25coder7b,humaneval,code_then_plan,TEST_FAIL,11
6,qwen25coder7b,humaneval,code_then_plan,EXEC_FAIL,1
10,qwen25coder7b,humaneval,code_then_plan_repair,PASS,151


### 2-3. 복구 전략 효과 (recovered_by 분포)

In [10]:
if not df_traj.empty and "recovered_by" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy", "recovered_by"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

,model,dataset,strategy,recovered_by,count
0,qwen25coder7b,humaneval,code_then_plan,generate,116
1,qwen25coder7b,humaneval,code_then_plan,plan_code,36
2,qwen25coder7b,humaneval,code_then_plan_repair,generate,115
3,qwen25coder7b,humaneval,code_then_plan_repair,plan_code,34
4,qwen25coder7b,humaneval,code_then_plan_repair,repair,2
5,qwen25coder7b,humaneval,proposed_v1,generate,121
6,qwen25coder7b,humaneval,proposed_v1,plan_code,21
7,qwen25coder7b,humaneval,proposed_v1,repair,12
9,qwen25coder7b,humaneval,proposed_v2,plan_code,72
10,qwen25coder7b,humaneval,proposed_v2,repair,40


### 2-4. 초기 실패 → 최종 성공 전환율 (recovery rate)

In [11]:
if not df_traj.empty:
    # initial_failed 컬럼이 없으면 generate step 기준으로 직접 계산
    if "initial_failed" not in df_traj.columns:
        df_traj["initial_failed"] = df_traj.get("transition_path", pd.Series()).apply(
            lambda p: (p[0].split(":")[0] != "PASS") if isinstance(p, list) and p else None
        )

    if "final_status" in df_traj.columns:
        df_traj["final_passed"] = df_traj["final_status"] == "PASS"

    recovery = (
        df_traj[df_traj["initial_failed"] == True]
        .groupby(["model", "dataset", "strategy"])["final_passed"]
        .agg(recovered="sum", total="count")
    )
    recovery["recovery_rate"] = recovery["recovered"] / recovery["total"]
    display(recovery.round(4))

recovered  total  recovery_rate
model         dataset   strategy                                    
qwen25coder7b humaneval proposed_v2        124    135         0.9185
              mbpp      proposed_v2         87    121         0.7190

## 3. step_logs.jsonl 로드 → call 수준 분석

In [12]:
def load_step_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "step_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_steps = load_step_logs(runs)
print(f"step rows: {len(df_steps)}")
df_steps.head(3)

step rows: 7541


/tmp/ipykernel_3467007/584948508.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


,run_id,dataset,problem_id,method,trajectory_id,step_id,call_index,candidate_id,stage,is_retry,is_repair,is_planner,input_tokens,output_tokens,total_tokens,latency_sec,avg_entropy,max_entropy,entropy_std,first_20pct_entropy,last_20pct_entropy,code,exec_ok,test_pass,status,error_type,error_stage,error_message,tests_passed,tests_total,code_length,selected,selection_rank,entry_point,model,strategy,policy_action,planner_output,policy_state,policy_stop,policy_stop_reason
0,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,0,0,0,generate,False,False,False,133,79,212,1.0578,0.2227,2.2908,0.4371,0.7167,0.0870,None,False,False,EXEC_FAIL:IndentationError,IndentationError,exec,"Traceback (most recent call last):\n File ""/h...",NaN,None,520,None,None,has_close_elements,phi35mini,repair,NaN,NaN,NaN,NaN,NaN
1,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,1,1,0,repair,False,True,False,529,184,713,1.8635,0.0201,1.4069,0.1418,0.0146,0.0768,None,True,False,TEST_FAIL:AssertionError,AssertionError,test,"Traceback (most recent call last):\n File ""/h...",NaN,None,371,None,None,has_close_elements,phi35mini,repair,NaN,NaN,NaN,NaN,NaN
2,phase2_phi35mini_humaneval_repair_0506103313,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,2,2,0,repair,False,True,False,463,187,650,1.8770,0.0420,2.1760,0.2121,0.0371,0.1137,None,True,False,TEST_FAIL:AssertionError,AssertionError,test,"Traceback (most recent call last):\n File ""/h...",NaN,None,371,None,None,has_close_elements,phi35mini,repair,NaN,NaN,NaN,NaN,NaN


### 3-1. stage별 PASS 비율 (call 수준 성공률)

In [13]:
if not df_steps.empty and "stage" in df_steps.columns and "status" in df_steps.columns:
    df_steps["passed"] = df_steps["status"] == "PASS"
    stage_stats = (
        df_steps[df_steps["stage"].isin(["generate", "repair", "plan_code"])]
        .groupby(["model", "dataset", "strategy", "stage"])["passed"]
        .agg(pass_count="sum", total="count")
    )
    stage_stats["pass_rate"] = stage_stats["pass_count"] / stage_stats["total"]
    display(stage_stats.round(4))

pass_count  total  \
model         dataset   strategy              stage                          
phi35mini     humaneval repair                generate            6    164   
                                              repair             88   1427   
                        single                generate            6    164   
qwen25coder7b humaneval code_then_plan        generate          116    164   
                                              plan_code          36    164   
                        code_then_plan_repair generate          115    164   
                                              plan_code          34    119   
                                              repair              2     85   
                        proposed_v1           generate          121    164   
                                              plan_code          21     76   
                                              repair             12     77   
                        proposed_v2           generate           29    164   
                                              plan_code          84    191   
                                              repair             40    197   
                        repair                generate          113    164   
                                              repair             25    554   
                        single                generate          116    164   
              mbpp      code_then_plan        generate          139    257   
                                              plan_code          74    493   
                        proposed_v2           generate          136    257   
                                              plan_code          85    394   
                                              repair              2    244   
                        single                generate          135    257   

                                                         pass_rate  
model         dataset   strategy              stage                 
phi35mini     humaneval repair                generate      0.0366  
                                              repair        0.0617  
                        single                generate      0.0366  
qwen25coder7b humaneval code_then_plan        generate      0.7073  
                                              plan_code     0.2195  
                        code_then_plan_repair generate      0.7012  
                                              plan_code     0.2857  
                                              repair        0.0235  
                        proposed_v1           generate      0.7378  
                                              plan_code     0.2763  
                                              repair        0.1558  
                        proposed_v2           generate      0.1768  
                                              plan_code     0.4398  
                                              repair        0.2030  
                        repair                generate      0.6890  
                                              repair        0.0451  
                        single                generate      0.7073  
              mbpp      code_then_plan        generate      0.5409  
                                              plan_code     0.1501  
                        proposed_v2           generate      0.5292  
                                              plan_code     0.2157  
                                              repair        0.0082  
                        single                generate      0.5253

### 3-2. error_type 분포 (실패 원인 분석)

In [14]:
if not df_steps.empty and "error_type" in df_steps.columns:
    failed_steps = df_steps[
        df_steps["status"].notna()
        & (df_steps["status"] != "PASS")
        & (df_steps["status"] != "PLAN_DONE")
    ]
    display(
        failed_steps.groupby(["model", "dataset", "strategy", "error_type"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
        .head(60)
    )

,model,dataset,strategy,error_type,count
0,phi35mini,humaneval,repair,AssertionError,1165
2,phi35mini,humaneval,repair,IndentationError,148
6,phi35mini,humaneval,repair,SyntaxError,74
7,phi35mini,humaneval,repair,TypeError,39
8,phi35mini,humaneval,repair,ValueError,22
3,phi35mini,humaneval,repair,IndexError,20
5,phi35mini,humaneval,repair,RecursionError,18
4,phi35mini,humaneval,repair,NameError,10
1,phi35mini,humaneval,repair,AttributeError,1
10,phi35mini,humaneval,single,IndentationError,150


### 3-3. 토큰 사용량 (step 기준)

In [15]:
if not df_steps.empty and "total_tokens" in df_steps.columns:
    display(
        df_steps.groupby(["model", "dataset", "strategy", "stage"])["total_tokens"]
        .agg(["mean", "median", "sum"])
        .round(1)
        .sort_index()
    )

mean    median  \
model         dataset   strategy              stage                           
phi35mini     humaneval repair                generate   468.4000  559.5000   
                                              repair     940.9000  846.0000   
                        single                generate   471.2000  568.5000   
qwen25coder7b humaneval code_then_plan        generate   298.4000  277.0000   
                                              plan       283.7000  271.5000   
                                              plan_code  409.9000  386.0000   
                        code_then_plan_repair generate   303.2000  273.0000   
                                              plan       296.0000  275.0000   
                                              plan_code  428.7000  394.0000   
                                              repair     575.9000  461.0000   
                        proposed_v1           generate   299.2000  285.0000   
                                              plan       481.2000  448.5000   
                                              plan_code  442.0000  407.0000   
                                              repair     947.1000  972.0000   
                        proposed_v2           generate   245.2000  182.5000   
                                              plan       288.9000  276.0000   
                                              plan_code  447.2000  407.0000   
                                              repair    1028.2000 1030.0000   
                                              replan     740.3000  682.0000   
                        repair                generate   305.2000  283.0000   
                                              repair     946.3000  920.5000   
                        single                generate   300.9000  273.5000   
              mbpp      code_then_plan        generate   205.4000  179.0000   
                                              plan       209.4000  203.0000   
                                              plan_code  324.0000  304.0000   
                        proposed_v2           generate   199.1000  180.0000   
                                              plan       205.5000  195.0000   
                                              plan_code  372.0000  337.5000   
                                              repair     855.2000  806.5000   
                                              replan     565.4000  524.0000   
                        single                generate   200.1000  180.0000   

                                                             sum  
model         dataset   strategy              stage               
phi35mini     humaneval repair                generate     76824  
                                              repair     1342602  
                        single                generate     77285  
qwen25coder7b humaneval code_then_plan        generate     48944  
                                              plan         46519  
                                              plan_code    67218  
                        code_then_plan_repair generate     49730  
                                              plan         35223  
                                              plan_code    51014  
                                              repair       48953  
                        proposed_v1           generate     49075  
                                              plan         36575  
                                              plan_code    33590  
                                              repair       72925  
                        proposed_v2           generate     40212  
                                              plan         27733  
                                              plan_code    85414  
                                              repair      202553  
                                              replan       70325  
                        repair                

## 4. 전체 성능 요약 테이블 (논문/보고서용)

In [16]:
REPORT_COLS = [
    "model", "dataset", "strategy",
    "success_at_k", "execution_success_rate", "conditional_success",
    "code_failed", "define_test_failed", "run_test_failed",
]
existing = [c for c in REPORT_COLS if c in df_summary.columns]
df_report = (
    df_summary[existing]
    .sort_values(["dataset", "model", "success_at_k"], ascending=[True, True, False])
    .reset_index(drop=True)
)
display(df_report)

,model,dataset,strategy,success_at_k,execution_success_rate,conditional_success,code_failed,define_test_failed,run_test_failed
0,phi35mini,humaneval,repair,0.5732,0.9451,0.6065,9,0,61
1,phi35mini,humaneval,single,0.0366,0.0671,0.5455,153,0,5
2,qwen25coder7b,humaneval,proposed_v1,0.9390,0.9939,0.9448,1,0,9
3,qwen25coder7b,humaneval,proposed_v2,0.9329,0.9939,0.9387,1,0,10
4,qwen25coder7b,humaneval,code_then_plan,0.9268,0.9939,0.9325,0,0,0
5,qwen25coder7b,humaneval,code_then_plan_repair,0.9207,0.9756,0.9437,0,0,0
6,qwen25coder7b,humaneval,repair,0.8415,0.8780,0.9583,20,0,6
7,qwen25coder7b,humaneval,single,0.7073,0.8841,0.8000,19,0,29
8,qwen25coder7b,mbpp,proposed_v2,0.8677,0.9844,0.8814,4,0,30
9,qwen25coder7b,mbpp,code_then_plan,0.8288,0.9961,0.8320,0,0,0


In [ ]:
# CSV로 저장
out_path = Path("results_report.csv")
df_report.to_csv(out_path, index=False)
print(f"저장 완료: {out_path.resolve()}")